# Fase 7 — Addestramento PPO preliminare

## Obiettivo

La Fase 7 introduce un primo agente di Reinforcement Learning basato su **PPO (Proximal Policy Optimization)**, integrato con l'ambiente di trading sviluppato nelle fasi precedenti.

L'obiettivo di questa fase non è ancora dimostrare la redditività della strategia, ma verificare che il processo di training funzioni end-to-end, che le principali metriche PPO siano monitorate e che la policy produca azioni non degenerative e ragionevolmente riproducibili su seed differenti.

## Implementazione

La Fase 7 è stata integrata direttamente nella struttura esistente del progetto, senza creare una pipeline RL separata.

Le componenti principali introdotte sono:

- `scripts/train_ppo.py`: entry point per il training PPO;
- `src/gold_rl/rl/callbacks.py`: callback per metriche, distribuzione delle azioni e checkpoint;
- configurazione PPO integrata in `config.yaml`;
- dipendenze Stable-Baselines3, PyTorch e TensorBoard integrate in `requirements.txt`;
- `models/prototype/`: salvataggio dei modelli e checkpoint;
- `logs/prototype/`: log di training e metriche delle azioni;
- `reports/ppo_prototype/`: riepiloghi delle esecuzioni.

L'agente riutilizza il `TradingEnv` della Fase 6 e lo stesso motore di execution/portfolio delle baseline, mantenendo quindi coerenti costi di transazione, spread, slippage e turnover.

## Configurazione del training

Il prototipo utilizza **PPO con policy MLP**. I principali parametri sono:

| Parametro | Valore |
|---|---:|
| Learning rate | 0.0003 |
| `n_steps` | 2048 |
| Batch size | 256 |
| Epoch per update | 10 |
| Gamma | 0.99 |
| GAE lambda | 0.95 |
| Clip range | 0.20 |
| Hidden layers | 128, 128 |
| Timesteps nominali | 250,000 |
| Seed | 42, 43, 44 |

Durante il training vengono monitorati, tra gli altri, `policy_gradient_loss`, `value_loss`, `entropy_loss`, `approx_kl`, `explained_variance`, reward degli episodi e distribuzione delle azioni.

Le azioni dell'ambiente sono `0 = short`, `1 = flat`, `2 = long`. La soglia utilizzata per identificare una policy fortemente degenerata è il 99% su una singola azione.

## Training multi-seed

Il training preliminare è stato eseguito con tre seed indipendenti: **42, 43 e 44**.

Ogni esecuzione ha raggiunto **251.904 timestep effettivi**. Il valore è leggermente superiore ai 250.000 timestep nominali perché PPO procede per rollout di dimensione fissa (`n_steps = 2048`).

Per ciascun seed sono stati registrati 50 episodi di training.

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {'seed': 42, 'timesteps': 251904, 'episodes': 50, 'mean_reward': -0.02994002,
     'last_reward': -0.036439, 'short': 0.4240742505, 'flat': 0.2995744411,
     'long': 0.2763513084, 'degenerate': False},
    {'seed': 43, 'timesteps': 251904, 'episodes': 50, 'mean_reward': -0.01746424,
     'last_reward': -0.019003, 'short': 0.3802162729, 'flat': 0.2887607978,
     'long': 0.3310229294, 'degenerate': False},
    {'seed': 44, 'timesteps': 251904, 'episodes': 50, 'mean_reward': -0.01324006,
     'last_reward': -0.008955, 'short': 0.3576084540, 'flat': 0.4037411077,
     'long': 0.2386504383, 'degenerate': False},
])

results.style.format({
    'mean_reward': '{:.6f}',
    'last_reward': '{:.6f}',
    'short': '{:.2%}',
    'flat': '{:.2%}',
    'long': '{:.2%}'
})

## Risultati

| Seed | Mean reward | Last reward | Short | Flat | Long | Degenerata |
|---:|---:|---:|---:|---:|---:|---|
| 42 | -0.029940 | -0.036439 | 42.41% | 29.96% | 27.64% | No |
| 43 | -0.017464 | -0.019003 | 38.02% | 28.88% | 33.10% | No |
| 44 | -0.013240 | -0.008955 | 35.76% | 40.37% | 23.87% | No |

Tutti e tre i seed utilizzano in modo significativo le tre azioni disponibili. Non è stato osservato un collasso verso una singola posizione, quindi il criterio di **non-degenerazione della policy** è soddisfatto nel training preliminare.

I reward rimangono negativi per tutti i seed. La Fase 7 non fornisce quindi evidenza di redditività: il risultato positivo di questa fase riguarda soprattutto la corretta integrazione dell'agente PPO, la stabilità operativa del training e l'assenza di una degenerazione estrema delle azioni.

In [ ]:
import matplotlib.pyplot as plt

action_dist = results.set_index('seed')[['short', 'flat', 'long']]
ax = action_dist.plot(kind='bar', figsize=(8, 4))
ax.set_title('Distribuzione delle azioni per seed')
ax.set_xlabel('Seed')
ax.set_ylabel('Frequenza relativa')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Conclusioni

La Fase 7 ha completato il primo training PPO multi-seed sulla pipeline di trading.

Le tre esecuzioni hanno raggiunto circa 252k timestep ciascuna senza errori e senza degenerazione della policy. Le distribuzioni delle azioni differiscono tra seed, ma short, flat e long rimangono tutte rappresentate in misura sostanziale.

Il criterio preliminare richiesto — **azioni non degenerative e comportamento riproducibile entro una variabilità ragionevole** — risulta quindi supportato dai tre training eseguiti.

I reward negativi indicano tuttavia che questi risultati devono essere interpretati come validazione della pipeline RL e non come validazione economica della strategia. Le fasi successive dovranno concentrarsi sull'analisi delle performance out-of-sample, sul confronto con le baseline e sull'eventuale tuning dell'agente.